In [1]:
!pip install -q transformers sentence-transformers datasets accelerate bitsandbytes openpyxl
!pip install -q langchain langchain_community langchain_core langchain_classic
!pip install -q langchain_huggingface langchain_ollama huggingface_hub faiss-cpu
!pip install -q fastapi uvicorn streamlit requests pyngrok pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 87.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 97.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 139.6 MB/s eta 0:00:0000:01


In [13]:
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install langchain-ollama

import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

!ollama pull llama3.2:3B

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.7.0-6).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [4]:
import os
import time
import subprocess
import requests

%pip install -q fastapi uvicorn streamlit requests pyngrok python-dotenv

# Resolve project files from a src-based layout in Colab/runtime
workspace_candidates = [
    "/content",
    "/content/LLM_Project",
    "/content/drive/MyDrive/LLM_Project",
]
src_candidates = [os.path.join(p, "src") for p in workspace_candidates]
project_dir = next((p for p in src_candidates if os.path.exists(os.path.join(p, "api_server.py"))), None)
if project_dir is None:
    raise FileNotFoundError("src/api_server.py not found. Upload/sync project files to Colab first.")

if not os.path.exists(os.path.join(project_dir, "streamlit_app.py")):
    raise FileNotFoundError("src/streamlit_app.py not found in the same project folder.")

print(f"Using project directory: {project_dir}")

# Load .env from src first, then project root (if present)
from dotenv import load_dotenv

env_candidates = [
    os.path.join(project_dir, ".env"),
    os.path.join(os.path.dirname(project_dir), ".env"),
]
env_path = next((p for p in env_candidates if os.path.exists(p)), None)
if env_path:
    load_dotenv(env_path, override=False)
    print(f"Loaded env vars from: {env_path}")
else:
    print("No .env file found in src or project directory; relying on runtime env vars.")

# Restart API if already running
if "api_proc" in globals() and api_proc is not None and api_proc.poll() is None:
    api_proc.terminate()
    time.sleep(1)

# Start API
api_proc = subprocess.Popen(
    ["uvicorn", "api_server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=project_dir,
)

# Wait for API health
api_ready = False
for _ in range(20):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).ok:
            api_ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(1)

if not api_ready:
    raise RuntimeError("API failed to start. Check src/api_server.py and dependencies.")

# Restart Streamlit if already running
if "st_proc" in globals() and st_proc is not None and st_proc.poll() is None:
    st_proc.terminate()
    time.sleep(1)

# Start Streamlit UI
st_proc = subprocess.Popen(
    [
        "streamlit", "run", "streamlit_app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--browser.serverAddress", "0.0.0.0",
    ],
    cwd=project_dir,
)
time.sleep(5)

from pyngrok import ngrok

token = os.environ.get("NGROK_AUTHTOKEN") or os.environ.get("NGROK_TOKEN")
if not token:
    raise RuntimeError(
        "NGROK token is missing. Put it in .env (NGROK_AUTHTOKEN=... or NGROK_TOKEN=...) or run: %env NGROK_AUTHTOKEN=your_token_here"
    )
ngrok.set_auth_token(token)

if "ngrok_tunnel" in globals() and ngrok_tunnel is not None:
    try:
        ngrok.disconnect(ngrok_tunnel.public_url)
    except Exception:
        pass

ngrok_tunnel = ngrok.connect(8501)
print("Open this app URL:", ngrok_tunnel.public_url)
print("In Streamlit sidebar, API URL should be: http://127.0.0.1:8000/query")

Using project directory: /content
Loaded env vars from: /content/.env
Open this app URL: https://phenomenally-slavish-lilly.ngrok-free.dev                                
In Streamlit sidebar, API URL should be: http://127.0.0.1:8000/query


In [10]:
import os
import sys

# Make local src files importable in Colab
candidate_dirs = [
    "./src",
    "/content/src",
    "/content/LLM_Project/src",
    "/content/drive/MyDrive/LLM_Project/src",
]
for path in candidate_dirs:
    if path not in sys.path and os.path.exists(path):
        sys.path.insert(0, path)

# Shared model/prompt pipeline + shared guardrails module
import llm_llama
from guardrails import (
    default_guardrails,
    is_clearly_out_of_scope,
    has_sufficient_context,
    contains_unsupported_numbers,
)

llm_llama.refresh_rag_resources()


def guardrailed_answer(query: str, k: int = 5) -> str:
    if is_clearly_out_of_scope(query, default_guardrails):
        return default_guardrails.out_of_domain_reply

    docs = llm_llama.vec_db.similarity_search(query, k=k)
    contexts = [d.page_content for d in docs]

    if not has_sufficient_context(contexts, default_guardrails):
        return default_guardrails.insufficient_context_reply

    context_text = "\n\n".join(contexts)
    result = llm_llama.qa_chain.invoke({"query": query})
    answer = result["result"] if isinstance(result, dict) and "result" in result else str(result)

    if contains_unsupported_numbers(answer, context_text):
        print("[guardrail warning] Answer may include numbers not explicitly present in retrieved context.")

    return answer

print("Guardrails + model loaded from src. Use guardrailed_answer('your question').")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Guardrails + model loaded. Use guardrailed_answer('your question').


In [21]:
import time
import textwrap

benchmark_query = "Provide me more information regarding NUST Freelancer Digital account."

start_time = time.perf_counter()
benchmark_answer = guardrailed_answer(benchmark_query)
elapsed_seconds = time.perf_counter() - start_time

print("Benchmark query:", benchmark_query)
print(f"Inference time (guardrailed_answer): {elapsed_seconds:.2f} seconds")
print("\nAnswer:")
print(textwrap.fill(benchmark_answer, width=100))

Benchmark query: Provide me more information regarding NUST Freelancer Digital account.
Inference time (guardrailed_answer): 12.76 seconds

Answer:
I can only help with NUST Bank product and app questions related to the provided context.   Would
you like me to provide more details about the NUST Freelancer Digital Account benefits or is there
something else I can assist you with?


In [7]:
!ollama ps

/bin/bash: line 1: ollama: command not found


In [11]:
!nvidia-smi

Sun Apr  5 08:32:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             28W /   70W |    3325MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
import time
import subprocess
import requests

!pip install -q fastapi uvicorn streamlit requests pyngrok python-dotenv

# Resolve project files from a src-based layout in Colab/runtime
workspace_candidates = [
    "/content",
    "/content/LLM_Project",
    "/content/drive/MyDrive/LLM_Project",
]
src_candidates = [os.path.join(p, "src") for p in workspace_candidates]
project_dir = next((p for p in src_candidates if os.path.exists(os.path.join(p, "api_server.py"))), None)
if project_dir is None:
    raise FileNotFoundError("src/api_server.py not found. Upload/sync project files to Colab first.")

if not os.path.exists(os.path.join(project_dir, "streamlit_app.py")):
    raise FileNotFoundError("src/streamlit_app.py not found in the same project folder.")

print(f"Using project directory: {project_dir}")

# Load .env from src first, then project root (if present)
from dotenv import load_dotenv

env_candidates = [
    os.path.join(project_dir, ".env"),
    os.path.join(os.path.dirname(project_dir), ".env"),
]
env_path = next((p for p in env_candidates if os.path.exists(p)), None)
if env_path:
    load_dotenv(env_path, override=False)
    print(f"Loaded env vars from: {env_path}")
else:
    print("No .env file found in src or project directory; relying on runtime env vars.")

# Restart API if already running
if "api_proc" in globals() and api_proc is not None and api_proc.poll() is None:
    api_proc.terminate()
    time.sleep(1)

# Start API
api_proc = subprocess.Popen(
    ["uvicorn", "api_server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=project_dir,
)

# Wait for API health
api_ready = False
for _ in range(20):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).ok:
            api_ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(1)

if not api_ready:
    raise RuntimeError("API failed to start. Check src/api_server.py and dependencies.")

# Restart Streamlit if already running
if "st_proc" in globals() and st_proc is not None and st_proc.poll() is None:
    st_proc.terminate()
    time.sleep(1)

# Start Streamlit UI
st_proc = subprocess.Popen(
    [
        "streamlit", "run", "streamlit_app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--browser.serverAddress", "0.0.0.0",
    ],
    cwd=project_dir,
)
time.sleep(5)

from pyngrok import ngrok

token = os.environ.get("NGROK_AUTHTOKEN") or os.environ.get("NGROK_TOKEN")
if not token:
    raise RuntimeError(
        "NGROK token is missing. Put it in .env (NGROK_AUTHTOKEN=... or NGROK_TOKEN=...) or run: %env NGROK_AUTHTOKEN=your_token_here"
    )
ngrok.set_auth_token(token)

if "ngrok_tunnel" in globals() and ngrok_tunnel is not None:
    try:
        ngrok.disconnect(ngrok_tunnel.public_url)
    except Exception:
        pass

ngrok_tunnel = ngrok.connect(8501)
print("Open this app URL:", ngrok_tunnel.public_url)
print("In Streamlit sidebar, API URL should be: http://127.0.0.1:8000/query")

Using project directory: /content
Loaded env vars from: /content/.env


Open this app URL: https://phenomenally-slavish-lilly.ngrok-free.dev
In Streamlit sidebar, API URL should be: http://127.0.0.1:8000/query
